In [1]:
include("CRD_STA.jl")
include("LST_BEK.jl")
import .CRD_BF as BF
using Plots
using LinearAlgebra
using ProgressMeter
using NonlinearEigenproblems

In [ ]:
function rayleigh_quotient_iteration(A, B, sigma; q0=rand(size(A, 1), 1))

    flg = true
    while flg
        sigma0 = sigma[1]+ 0.0e0im
        q = (A - sigma*B) \ (B*q0)
        q0 = q/maximum(abs.(q))
        sigma = ((q0'*(A*q0))/(q0'*(B*q0)))[1]
        if abs(sigma-sigma0)<=eps(1.0f0)
            flg = false
        end

    end

      return sigma, q0
    end

In [ ]:
#Critical Re=215 be=0.08

In [ ]:
N_cheb = 99
Re = 100
R = sqrt(Re)
Ma = 1
Mr = Re*Ma
Tw = 1.2
gamma = 1.4
sigma = 0.72
be = 0.07759
omega = 0
F,G,W,f,q,D,D2,x,t,PHI = baseflow_var(N_cheb)
rho,H,T = T_ca(Re,Ma,f,q,W,gamma,Tw,x,t,N_cheb)
lam = -(2/3) * T
kappa = (1/sigma) * T 

In [ ]:
al = 0.4
be = 0.02
A,B = Time_mode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,al,be)
C = eigen(A,B)
va = C.values
point = filter(x->0<imag(x)<0.01&&abs.(real(x)+be)>0.005 , va)

In [290]:
al = 0.4
dat = [empty! empty! empty!]
for be = 0.01:0.01:0.2
    A,B = Time_mode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,al,be)
    C = eigen(A,B)
    va = C.values
    point = filter(x->0<imag(x)<0.001&&abs.(real(x)+be)>0.005 , va)
    if point == []
        arr = [al be -1]
    else
        arr = [al be imag(point[1])]
    end
    dat = [dat;arr]
end

In [ ]:
point = filter(x->0<imag(x)<0.01, va1)
# point = filter(x->0<imag(x)<0.001&&abs.(real(x)+be)>0.005 , va)

In [ ]:
scatter(real(va),imag(va),xlims=[-1,1],ylims=[-0.1,0.1])
scatter!(real(va1),imag(va1),xlims=[-1,1],ylims=[-0.1,0.1])

In [ ]:
C0,C1,C2,C3,C4 = KEB_SpatialMode.KEB_LST_ALL("Vonkarmen.txt",N_cheb,omega,be,Re,1,2)
nep = PEP([C0,C1,C2,C3,C4]); #Create a PEP object
sc=10
nep1 = shift_and_scale(nep,scale=sc);
mult_scale = norm(nep1.A[end]);
nep2 = PEP(nep1.A ./ mult_scale);
λ1,v1 = iar(nep2,σ=0.05,neigs=50,maxit=500);
λ_orig = sc*λ1
point = filter(x -> -0.001 < imag(x) < 0.001, λ_orig)
# vec = select(point,λ_orig,v1)

In [ ]:
vec,u = select(point[1],λ_orig,v1)

In [ ]:
be = 0.1
A0,A1,A2 = Spatial_mode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be)
nep = PEP([A0,A1,A2]); 
sc=10
nep1 = shift_and_scale(nep,scale=sc);
mult_scale = norm(nep1.A[end]);
nep2 = PEP(nep1.A ./ mult_scale);
λ1,v1 = iar(nep2,σ = 0.03,neigs = 50,maxit = 500);
λ_orig = λ1 * sc
point = filter(x -> -0.02 < imag(x) < 0, λ_orig)

In [ ]:
function select(values,val,v1)
    index=findall(x->x==values,val)
    vec = v1[:,index]
    u = vec[1:N_cheb-1,1]
    v = vec[N_cheb:2*(N_cheb-1),1]
    return vec,u,v
end

In [ ]:
vec,u,v= select(  0.06018121067091271 - 0.01890147262570327im

,λ_orig,v1)
u = [0;u;0]
v = [0;v;0]

In [ ]:
plot(x,abs.(u))

In [ ]:
scatter(real(λ_orig),imag(λ_orig),xlim=[-0.5,1],xlabel="α_r",ylabel="α_i")